In [23]:
# 25년 9월 17일 수요일 화이팅입니다!!
'''
오늘은 머신러닝 모델들의 성능 평가 방식 원리에 대해 공부해보겠다.
'''

'\n오늘은 머신러닝 모델들의 성능 평가 방식 원리에 대해 공부해보겠다.\n'

## 1. 분류모델(Classifier) 평가 지표

In [24]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split


from sklearn.metrics      import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot  as plt

In [25]:
data = pd.read_csv('datasets/heart.csv')

print(data.head(), data.shape, sep='\n')

   age  sex  cp  trtbps  chol  fbs  ...  exng  oldpeak  slp  caa  thall  output
0   63    1   3     145   233    1  ...     0      2.3    0    0      1       1
1   37    1   2     130   250    0  ...     0      3.5    0    0      2       1
2   41    0   1     130   204    0  ...     0      1.4    2    0      2       1
3   56    1   1     120   236    0  ...     0      0.8    2    0      2       1
4   57    0   0     120   354    0  ...     1      0.6    2    0      2       1

[5 rows x 14 columns]
(303, 14)


In [26]:
x = data.drop(['output'],axis=1)
y = data['output']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state= 27)

In [27]:
model = RandomForestClassifier(n_estimators=1000,    # 트리 개수(기본 100)
                               criterion='gini',    # 불순도 측정 (gini or entropy)
                               max_depth=None,      # 트리 최대 깊이
                               min_samples_split=2, # 노드 분할의 최소 샘플 수
                               min_samples_leaf=1,  # 리프 노드가 가져야 할 최소 샘플
                               max_features='sqrt', # 트리 분할시 최대 특성 개수
                               bootstrap=True,      # 배깅 방식 적용 여부
                               oob_score=True,      # Out Of Bag 샘플로 정확도 평가 여부
                               n_jobs=1,            # 병렬 처리 여부
                               random_state=28,
                               verbose=1            # 학습 과정 출력 여부
                               )

model.fit(x_train, y_train)

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.9s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    2.3s
[Parallel(n_jobs=1)]: Done 799 tasks      | elapsed:    4.2s


RandomForestClassifier(n_estimators=1000, n_jobs=1, oob_score=True,
                       random_state=28, verbose=1)

In [28]:
y_pred = model.predict(x_test)

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 799 tasks      | elapsed:    0.1s


### 혼동 행렬 (Confusion Matrix) - F1 score

In [29]:
'''
혼동 행렬은 분류모델(Classifier)의 예측 결과를 실제 정답과 비교해
예측의 정확성과 오류를 한눈에 볼 수 있도록 정리한 표이다.

칼럼에 예측을, 인덱스에 실제 결과를 보여주는 2*2 매트릭스로 구성
'''

'\n혼동 행렬은 분류모델(Classifier)의 예측 결과를 실제 정답과 비교해\n예측의 정확성과 오류를 한눈에 볼 수 있도록 정리한 표이다.\n\n칼럼에 예측을, 인덱스에 실제 결과를 보여주는 2*2 매트릭스로 구성\n'

In [30]:
confu = confusion_matrix(y_test, y_pred)

print(confu)
print("TP FN\nFP TN")
print('''\
TP (True Positive): 양성을 양성으로 올바르게 예측
TN (True Negative): 음성을 음성으로 올바르게 예측
FP (False Positive): 음성을 양성으로 잘못 예측 (1종 오류)
FN (False Negative): 양성을 음성으로 잘못 예측 (2종 오류)\
      ''')

[[37  5]
 [10 39]]
TP FN
FP TN
TP (True Positive): 양성을 양성으로 올바르게 예측
TN (True Negative): 음성을 음성으로 올바르게 예측
FP (False Positive): 음성을 양성으로 잘못 예측 (1종 오류)
FN (False Negative): 양성을 음성으로 잘못 예측 (2종 오류)      


In [31]:
'''
Accuracy(정확도) : (TP + TN) / 전체 case
    but accuracy의 문제점 - TP와 TN의 비율을 고려하지 않음
                            즉 Positive를 예측하는 능력이 포함되지 않음

Precision(정밀도) : TP / (TP + FP)
    모델이 Positive로 예측한 것 가운데 적중률

Recall(재현율) : TP / (TP + FN)
    실제 Positive 가운데 모델이 적중시킨 것들의 비율
'''

'\nAccuracy(정확도) : (TP + TN) / 전체 case\n    but accuracy의 문제점 - TP와 TN의 비율을 고려하지 않음\n                            즉 Positive를 예측하는 능력이 포함되지 않음\n\nPrecision(정밀도) : TP / (TP + FP)\n    모델이 Positive로 예측한 것 가운데 적중률\n\nRecall(재현율) : TP / (TP + FN)\n    실제 Positive 가운데 모델이 적중시킨 것들의 비율\n'

In [32]:
'''
F1-score
Precision과 Recall을 동시에 고려하기 위해 고안된 것이 F1-Score
분류 모델에서 주로 사용하는 성능 지표이다.

F1 = 2 * (P * R) / (P + R)
'''

'\nF1-score\nPrecision과 Recall을 동시에 고려하기 위해 고안된 것이 F1-Score\n분류 모델에서 주로 사용하는 성능 지표이다.\n\nF1 = 2 * (P * R) / (P + R)\n'

### ROC와 AUC

In [33]:
'''
ROC : Receiver Operating Characteristic
    ROC 곡선은 다양한 임계값에서 다음 두 지표를 그래프로 나타낸 것: 
    X축: FPR (False Positive Rate) = FP / (FP + TN) = 1 - 특이도
    Y축: TPR (True Positive Rate) = TP / (TP + FN) = 재현율(민감도)


AUC : Area Under Curve
    ROC 곡선 아래의 면적을 의미(클수록 좋음)

    0.5~1.0 사이의 값을 가짐
    AUC = 1.0: 완벽한 분류기
    AUC = 0.9~1.0: 매우 우수한 성능
    AUC = 0.8~0.9: 좋은 성능
    AUC = 0.7~0.8: 괜찮은 성능
    AUC = 0.6~0.7: 약한 성능
    AUC = 0.5: 무작위 추측과 동일 (동전 던지기)
    AUC < 0.5: 무작위보다 나쁨 (예측을 반대로 하면 개선됨)
    
지니계수 : AUC에서 파생된 값
    Gini = 2 * AUC - 1
    0이면 모델이 무작위 예측 수준
    1이면 완벽한 분류기
'''

'\nROC : Receiver Operating Characteristic\n    ROC 곡선은 다양한 임계값에서 다음 두 지표를 그래프로 나타낸 것: \n    X축: FPR (False Positive Rate) = FP / (FP + TN) = 1 - 특이도\n    Y축: TPR (True Positive Rate) = TP / (TP + FN) = 재현율(민감도)\n\n\nAUC : Area Under Curve\n    ROC 곡선 아래의 면적을 의미(클수록 좋음)\n\n    0.5~1.0 사이의 값을 가짐\n    AUC = 1.0: 완벽한 분류기\n    AUC = 0.9~1.0: 매우 우수한 성능\n    AUC = 0.8~0.9: 좋은 성능\n    AUC = 0.7~0.8: 괜찮은 성능\n    AUC = 0.6~0.7: 약한 성능\n    AUC = 0.5: 무작위 추측과 동일 (동전 던지기)\n    AUC < 0.5: 무작위보다 나쁨 (예측을 반대로 하면 개선됨)\n    \n지니계수 : AUC에서 파생된 값\n    Gini = 2 * AUC - 1\n    0이면 모델이 무작위 예측 수준\n    1이면 완벽한 분류기\n'

## 2. 회귀(Regression)모델 평가지표

### MAE - Mean Absolute Error

In [34]:
'''
평균 절대 오차 - 오차 값들의 절댓값 평균

단점 - 모든 오차값을 동일한 중요도로 취급.
       큰 오차와 작은 오차들을 동일하게 취급함
'''

'\n평균 절대 오차 - 오차 값들의 절댓값 평균\n\n단점 - 모든 오차값을 동일한 중요도로 취급.\n       큰 오차와 작은 오차들을 동일하게 취급함\n'

### MSE - Mean Squared Error

In [35]:
'''
평균 제곱 오차
오차를 제곱한 뒤 평균을 계산하는 지표
제곱하기 때문에 큰 오차에 더 큰 가중치가 실린다.

단점 - 오차를 제곱하므로 결과값의 단위가 원본 데이터와 다르다.
'''

'\n평균 제곱 오차\n오차를 제곱한 뒤 평균을 계산하는 지표\n제곱하기 때문에 큰 오차에 더 큰 가중치가 실린다.\n\n단점 - 오차를 제곱하므로 결과값의 단위가 원본 데이터와 다르다.\n'

### RMSE - Root Mean Squared Error

In [36]:
'''
평균 제곱근 오차
간단히 말해 MSE의 제곱근이다.
MSE의 단점인 scale 차이가 해결
마찬가지로 큰 오차에 큰 가중치가 부여된다.

MSE와 RMSE 모두 큰 오차에 과적합될 수가 있다.
'''

'\n평균 제곱근 오차\n간단히 말해 MSE의 제곱근이다.\nMSE의 단점인 scale 차이가 해결\n마찬가지로 큰 오차에 큰 가중치가 부여된다.\n\nMSE와 RMSE 모두 큰 오차에 과적합될 수가 있다.\n'

### R-Squared - 결정계수

In [37]:
'''
결정계수는 모델이 데이터를 얼마나 잘 설명하는가를 나타내는 지표.
다시 말해 이 모델을 사용함에 따라 달라진 잔차의 정도를 평가함
R_Squared = 1 - (잔차 제곱의 합 / 총 변동)


모델의 성능을 직관적으로 평가하나
다차원 회귀 모델의 경우 결정계수만 보면 과적합이 발생한다.

또한 종속변수 요소를 많이 넣을수록 결정계수가 올라가기 때문에
잘 사용하지 않는다
'''

'\n결정계수는 모델이 데이터를 얼마나 잘 설명하는가를 나타내는 지표.\n다시 말해 이 모델을 사용함에 따라 달라진 잔차의 정도를 평가함\nR_Squared = 1 - (잔차 제곱의 합 / 총 변동)\n\n\n모델의 성능을 직관적으로 평가하나\n다차원 회귀 모델의 경우 결정계수만 보면 과적합이 발생한다.\n\n또한 종속변수 요소를 많이 넣을수록 결정계수가 올라가기 때문에\n잘 사용하지 않는다\n'

In [38]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import LinearRegression
from sklearn.ensemble        import RandomForestRegressor
from xgboost                 import XGBRegressor
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score

In [39]:
wine_data = pd.read_csv('datasets/winequality-white.csv')
wine_data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,alcohol
0,7.0,0.27,0.36,20.7,45.00,45.0,170.0,1.001,3.00,0.45,6,"R$ 45.512,00"
1,6.3,0.30,0.34,1.6,49.00,14.0,132.0,994.000,3.30,0.49,6,"R$ 45.421,00"
2,8.1,0.28,0.40,6.9,0.05,30.0,97.0,9.951,3.26,0.44,6,"R$ 45.301,00"
3,7.2,0.23,0.32,8.5,58.00,47.0,186.0,9.956,3.19,0.40,6,"R$ 45.544,00"
4,7.2,0.23,0.32,8.5,58.00,47.0,186.0,9.956,3.19,0.40,6,"R$ 45.544,00"


In [40]:
x = wine_data.drop(['quality', 'alcohol'], axis=1)
y = wine_data['quality']

In [41]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=51)

In [42]:
li_reg = LinearRegression()
li_reg.fit(x_train, y_train)
y_pred = li_reg.predict(x_test)

In [43]:
# 평균 절대 오차
MAE = mean_absolute_error(y_test, y_pred)
print(MAE)

0.6322876157863192


In [44]:
# 평균 제곱 오차
MQE = mean_squared_error(y_test, y_pred)
print(MQE)

0.6872637245460638


In [45]:
# 결정계수
r2 = r2_score(y_test, y_pred)
print(r2)

# 0.11이면 설명력이 거의 없..

0.11018230355127401


In [46]:
rf = RandomForestRegressor(n_estimators=100,
                            random_state=52)
rf.fit(x_train, y_train)
y_pred = rf.predict(x_test)

In [47]:
MAE = mean_absolute_error(y_test, y_pred)
print(MAE)

0.4536836734693877


In [48]:
MSE = mean_squared_error(y_test, y_pred)
print(MSE)

0.40688785714285713


In [49]:
r2 = r2_score(y_test, y_pred)
print(r2)

0.4731920181078193


In [50]:
rf = XGBRegressor()

rf.fit(x_train, y_train)
y_pred = rf.predict(x_test)

In [51]:
MAE = mean_absolute_error(y_test, y_pred)
print(MAE)

0.465908408164978


In [52]:
MSE = mean_squared_error(y_test, y_pred)
print(MSE)

0.4420000910758972


In [53]:
r2 = r2_score(y_test, y_pred)
print(r2)

0.4277312755584717


In [54]:
# XGBoost는 분류모델도 됨